# 第5讲：描述统计、探索性分析与工程数据可视化

案例数据：DS-C05施工能耗与教学碳排日表。

本Notebook用于核对数据口径，计算统计摘要，检查分布、关系和异常候选。


## 使用说明

1. 保持Notebook与`data`文件夹的相对位置不变。
2. 依次运行数据核对、描述统计、分组、关系和异常候选单元格。
3. 统计结果必须同时说明分析对象、有效记录数、单位和分组口径。


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_file = Path('/System/Library/Fonts/STHeiti Medium.ttc')
if font_file.exists():
    font_manager.fontManager.addfont(str(font_file))
    plt.rcParams['font.family'] = font_manager.FontProperties(fname=str(font_file)).get_name()
else:
    plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

FOCUS_VARIABLE = 'carbon_kgco2e'
GROUP_FIELD = 'construction_stage'

def locate_data(filename):
    candidate = Path('data') / filename
    if candidate.exists():
        return candidate
    raise FileNotFoundError(f'未找到{filename}，请保持Notebook与data目录的相对位置。')

data_path = locate_data('DS-C05_施工能耗与教学碳排日表.csv')
print('Python:', sys.version.split()[0])
print('数据文件:', data_path.as_posix())
print('分析变量:', FOCUS_VARIABLE, '分组字段:', GROUP_FIELD)

## 1. 数据对象与质量核对

一行表示一个自然日中的一个作业区，业务键为`date + zone_id`。


In [ ]:
df = pd.read_csv(data_path, parse_dates=['date'])
quality_check = pd.DataFrame({
    'item': ['rows', 'columns', 'duplicate_date_zone', 'carbon_missing'],
    'value': [len(df), df.shape[1], df.duplicated(['date', 'zone_id']).sum(), df['carbon_kgco2e'].isna().sum()]
})
display(df.head())
quality_check

## 2. 中心、离散与分位数

案例摘要描述已界定的180天×3区有限范围，因此总体标准差使用`ddof=0`。


In [ ]:
def descriptive_summary(series):
    valid = series.dropna()
    q1 = valid.quantile(0.25, interpolation='linear')
    q3 = valid.quantile(0.75, interpolation='linear')
    return pd.Series({
        'effective_n': valid.size,
        'missing_n': series.isna().sum(),
        'mean': valid.mean(),
        'median': valid.median(),
        'std_population': valid.std(ddof=0),
        'q1': q1,
        'q3': q3,
        'iqr': q3 - q1,
        'min': valid.min(),
        'max': valid.max(),
    })

summary = descriptive_summary(df[FOCUS_VARIABLE]).to_frame('value')
summary.round(3)

## 3. 施工阶段或作业区的分组摘要

课堂操作：把`GROUP_FIELD`从`construction_stage`改为`zone_id`，再运行本节。


In [ ]:
stage_labels = {
    'site_preparation': '场地准备',
    'foundation': '基础施工',
    'structure': '主体结构',
    'enclosure_mep': '围护机电',
    'finishing_commissioning': '装修调试',
}

group_summary = (
    df.groupby(GROUP_FIELD, observed=True)[FOCUS_VARIABLE]
      .agg(effective_n='count', mean='mean', median='median',
           std_population=lambda s: s.std(ddof=0),
           q1=lambda s: s.quantile(0.25), q3=lambda s: s.quantile(0.75))
      .reset_index()
)
group_summary['iqr'] = group_summary['q3'] - group_summary['q1']
group_summary['group_label'] = group_summary[GROUP_FIELD].map(stage_labels).fillna(group_summary[GROUP_FIELD].astype(str))
group_summary[['group_label', 'effective_n', 'mean', 'median', 'std_population', 'iqr']].round(3)

In [ ]:
plot_data = [(str(group), values.dropna().to_numpy())
             for group, values in df.groupby(GROUP_FIELD, observed=True)[FOCUS_VARIABLE]]
labels = [stage_labels.get(group, group) for group, _ in plot_data]
values = [value for _, value in plot_data]

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.boxplot(values, tick_labels=labels, showfliers=True)
ax.set_ylabel('教学碳排量（kgCO2e）')
ax.set_title(f'教学碳排分组分布｜分组字段：{GROUP_FIELD}')
ax.tick_params(axis='x', rotation=20)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## 4. 变量关系

相关系数只使用两列都完整的记录；解释时同时查看有效记录数、散点和分组。


In [ ]:
pair = df[['equipment_hours', 'carbon_kgco2e', 'construction_stage']].dropna()
pearson_r = pair['equipment_hours'].corr(pair['carbon_kgco2e'], method='pearson')
spearman_r = pair['equipment_hours'].corr(pair['carbon_kgco2e'], method='spearman')
correlation_result = pd.DataFrame({
    'effective_n': [len(pair)],
    'pearson_r': [pearson_r],
    'spearman_r': [spearman_r],
})
correlation_result.round(6)

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.8))
for stage, part in pair.groupby('construction_stage', observed=True):
    ax.scatter(part['equipment_hours'], part['carbon_kgco2e'], s=24, alpha=0.62,
               label=stage_labels.get(stage, stage))
ax.set_xlabel('设备工时（h）')
ax.set_ylabel('教学碳排量（kgCO2e）')
ax.set_title(f'设备工时与教学碳排｜成对完整N={len(pair)}')
ax.legend(frameon=False, fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## 5. 异常候选与工程复核

在施工阶段×作业区内计算1.5×IQR候选，再与质量标记合并。候选状态保持待复核。


In [ ]:
work = df.copy()
groups = work.groupby(['construction_stage', 'zone_id'], observed=True)['carbon_kgco2e']
work['q1_group'] = groups.transform(lambda s: s.quantile(0.25))
work['q3_group'] = groups.transform(lambda s: s.quantile(0.75))
work['iqr_group'] = work['q3_group'] - work['q1_group']
work['lower_bound'] = work['q1_group'] - 1.5 * work['iqr_group']
work['upper_bound'] = work['q3_group'] + 1.5 * work['iqr_group']
work['iqr_candidate'] = (
    work['carbon_kgco2e'].notna() &
    ((work['carbon_kgco2e'] < work['lower_bound']) |
     (work['carbon_kgco2e'] > work['upper_bound']))
)
work['quality_candidate'] = work['data_quality_flag'].ne('ok')
work['candidate_union'] = work['iqr_candidate'] | work['quality_candidate']
work['weekday'] = work['date'].dt.dayofweek

candidate_summary = pd.DataFrame({
    'iqr_candidate_rows': [int(work['iqr_candidate'].sum())],
    'quality_candidate_rows': [int(work['quality_candidate'].sum())],
    'union_rows': [int(work['candidate_union'].sum())],
    'union_dates': [int(work.loc[work['candidate_union'], 'date'].nunique())],
    'sunday_low_iqr_rows': [int((work['iqr_candidate'] &
                                 (work['carbon_kgco2e'] < work['lower_bound']) &
                                 work['weekday'].eq(6)).sum())],
})
candidate_summary

## 6. 观察记录

每条观察都应能回到统计表、图形和原始记录。


### 学生填写

**观察对象与范围**：

**统计量或图形证据**：

**比较口径**：

**仍需核查的问题**：


## 7. 课后完成

1. 输出包含有效N、均值、中位数、标准差、Q1、Q3和IQR的统计表；
2. 生成一幅分布或分组图、一幅时间或关系图；
3. 写两条包含对象、证据、口径和待核查问题的观察。
